# 22-05 · Flask: маршруты, шаблоны и формы

Практика к разделу [«Первое веб-приложение на Flask»](../../site/chapters/glava-22/22-05-flask.html). Полный проект — `projects/flask/todo-app/app.py`.

## Про тестовый клиент в этом ноутбуке

Обычно Flask-приложение запускают командой `python app.py`, и оно ждёт запросы браузера бесконечно (`app.run()`). В автоматически выполняемом ноутбуке нет браузера и нет смысла ждать вечно, поэтому мы используем `app.test_client()`. Он формирует запрос внутри процесса и передаёт его приложению через тестовый интерфейс Flask/Werkzeug, не открывая сетевой порт и не запуская настоящий сервер, — та же логика обработки запроса, что и в реальном приложении, но без сети (раздел 22.33 сайта разбирает это различие подробнее).

## Рабочий пример — собираем маленькое приложение

In [1]:
from flask import Flask, redirect, render_template_string, request, url_for

app = Flask(__name__)
zadachi = ["Выучить основы Python", "Собрать сайт на Flask"]

SHABLON_GLAVNOJ = """
<h1>Мой список задач</h1>
<ul>
{% for zadacha in zadachi %}
  <li>{{ zadacha }}</li>
{% endfor %}
</ul>
"""

SHABLON_PRIVET = "<h1>Привет, {{ imya }}!</h1>"


@app.route("/")
def glavnaya():
    return render_template_string(SHABLON_GLAVNOJ, zadachi=zadachi)


@app.route("/privet/<imya>")
def privet(imya):
    return render_template_string(SHABLON_PRIVET, imya=imya)


@app.route("/dobavit", methods=["POST"])
def dobavit():
    novaya_zadacha = request.form.get("zadacha", "").strip()
    if novaya_zadacha:
        zadachi.append(novaya_zadacha)
    return redirect(url_for("glavnaya"))


client = app.test_client()
print("Приложение и тестовый клиент готовы.")

Приложение и тестовый клиент готовы.


## Эксперимент 1 — главная страница (GET /)

In [2]:
otvet = client.get("/")
telo = otvet.get_data(as_text=True)

print("Код ответа:", otvet.status_code)
print(telo)

Код ответа: 200

<h1>Мой список задач</h1>
<ul>

  <li>Выучить основы Python</li>

  <li>Собрать сайт на Flask</li>

</ul>


## Проверка результата

In [3]:
assert otvet.status_code == 200
assert "Выучить основы Python" in telo
assert "Собрать сайт на Flask" in telo
print("Верно: главная страница отдаёт список задач с кодом 200.")

Верно: главная страница отдаёт список задач с кодом 200.


## Эксперимент 2 — динамический маршрут /privet/<imya>

In [4]:
otvet2 = client.get("/privet/Ада")
telo2 = otvet2.get_data(as_text=True)

print(telo2)
assert "Привет, Ада!" in telo2
print("Верно: значение из адреса подставилось в шаблон.")

<h1>Привет, Ада!</h1>
Верно: значение из адреса подставилось в шаблон.


## Эксперимент 3 — отправка формы (POST /dobavit)

In [5]:
kolichestvo_do = len(zadachi)

otvet3 = client.post("/dobavit", data={"zadacha": "Прочитать книгу"})
print("Код ответа:", otvet3.status_code)   # 302 — редирект на главную
print("Задач было:", kolichestvo_do, "-> стало:", len(zadachi))

assert otvet3.status_code == 302
assert len(zadachi) == kolichestvo_do + 1
assert "Прочитать книгу" in zadachi
print("Верно: POST-запрос добавил новую задачу и вернул редирект.")

Код ответа: 302
Задач было: 2 -> стало: 3
Верно: POST-запрос добавил новую задачу и вернул редирект.


## Задание ★★ Самостоятельная задача

Отправьте форму с пустой задачей (`{"zadacha": "   "}`) и убедитесь, что список задач не изменился — как и в настоящем `app.py`.

In [6]:
kolichestvo_do2 = len(zadachi)
client.post("/dobavit", data={"zadacha": "   "})

assert len(zadachi) == kolichestvo_do2
print("Верно: пустая (только пробелы) задача не была добавлена.")

Верно: пустая (только пробелы) задача не была добавлена.
